In [ ]:
import numpy as np
import pandas as pd
from sklearn.covariance import LedoitWolf
from scipy.optimize import curve_fit
import torch
from scipy.stats import zscore


In [ ]:
# Step 0: 从 Excel 文件中读取脑区坐标和基因表达矩阵
coordinates_df = pd.read_csv(r"E:\aliyun_backup\muilt_disorders\01_altas\BNA246_Cerebellum_Stem_Asym.csv")
coordinates = coordinates_df[['MNI_X', 'MNI_Y', 'MNI_Z']].values  # 转换为 numpy 数组

gene_expression_df = pd.read_excel(r"E:\aliyun_backup\muilt_disorders\15_Mitochondrial\Mitochondrial.xlsx")  # 基因表达矩阵文件
gene_expression_df = gene_expression_df.drop(columns='label')
gene_names = gene_expression_df.columns  # 基因名作为列名

gene_expression_matrix = gene_expression_df.values  # 转换为 numpy 数组

In [ ]:
from scipy.spatial.distance import pdist, squareform

# Step 1: 计算每对脑区的欧氏距离矩阵
distance_matrix = squareform(pdist(coordinates, metric='euclidean'))

# Step 2: 对基因表达矩阵按行进行 z-score 标准化
z_gene_expression = np.apply_along_axis(zscore, 1, gene_expression_matrix)


In [ ]:
# Step 3: 将距离矩阵和基因表达矩阵转换为 PyTorch 张量，并移动到 GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
distance_matrix = torch.tensor(distance_matrix, dtype=torch.float32, device=device)
z_gene_expression = torch.tensor(z_gene_expression, dtype=torch.float32, device=device)



In [ ]:
upper_tri_indices = torch.triu_indices(distance_matrix.size(0), distance_matrix.size(1), offset=1)
distances = distance_matrix[upper_tri_indices[0], upper_tri_indices[1]]

In [ ]:
# ---- Step 0: 数据准备 ----
z_gene_expression_np = z_gene_expression.cpu().numpy()  # [246, 6]
num_regions, num_genes = z_gene_expression_np.shape
region_pairs = list(zip(upper_tri_indices[0].cpu().numpy(), upper_tri_indices[1].cpu().numpy()))
num_connections = len(region_pairs)

In [ ]:
# ---- Step 1: 构建全基因表达网络（精度矩阵） ----
model_full = LedoitWolf()
model_full.fit(z_gene_expression_np.T)
cov_full = model_full.covariance_
precision_full = np.linalg.inv(cov_full)

# 提取上三角连接值（即构成网络的连接强度）
full_conn_values = np.array([precision_full[i, j] for i, j in region_pairs])



In [ ]:
# ---- Step 2: 拟合距离 vs 连接强度的理论模型 ----
def exponential_decay(d, A, n, B):
    return A * np.exp(-d / n) + B

# 获取距离对应的连接
distances = distance_matrix[upper_tri_indices[0], upper_tri_indices[1]].cpu().numpy()
correlations = precision_full[upper_tri_indices[0], upper_tri_indices[1]]

# 拟合非线性函数
initial_params = [1.0, 50.0, 0.0]
popt, _ = curve_fit(exponential_decay, distances, full_conn_values, p0=initial_params)
A, n, B = popt
print(f"拟合参数 A: {A}, n: {n}, B: {B}")


In [ ]:
import matplotlib.pyplot as plt

# 可视化拟合效果
plt.figure(figsize=(8, 6))
plt.scatter(distances, correlations, s=1, color='gray', alpha=0.5, label='Data')
sorted_indices = np.argsort(distances)
plt.plot(distances[sorted_indices], exponential_decay(distances[sorted_indices], *popt), 
         color='red', linewidth=2, label='Fitted Curve')
plt.xlabel("Distance between regions (mm)")
plt.ylabel("Correlated gene expression (CGE)")
plt.title("CGE vs. Distance with Fitted Exponential Decay")
plt.legend()
plt.savefig(r'E:\aliyun_backup\muilt_disorders\15_Mitochondrial\Mitochondrial_CGE_vs_Distance.tif',dpi=300)
plt.show()

In [ ]:
# ---- Step 3: 构建 expected_matrix（理论连接矩阵） ----
expected_matrix = np.zeros((num_regions, num_regions))
for i in range(num_regions):
    for j in range(i + 1, num_regions):
        d = distance_matrix[i, j].item()
        expected = A * np.exp(-d / n) + B
        expected_matrix[i, j] = expected
        expected_matrix[j, i] = expected

# 构建 residual matrix（残差网络）：observed - expected
residual_matrix = precision_full - expected_matrix


In [ ]:

# ---- Step 4: 初始化 gene_contribution_df ----
gene_contribution_df = pd.DataFrame(np.zeros((num_connections, num_genes)), columns=gene_names)


In [ ]:

# ---- Step 5: LOFO 分析：对每个基因特征进行一次 leave-out ----
for g in range(num_genes):
    print(f"Processing gene {g+1}/{num_genes}")

    # 去掉第 g 个基因
    reduced_expression = np.delete(z_gene_expression_np, g, axis=1)

    try:
        model_lofo = LedoitWolf()
        model_lofo.fit(reduced_expression.T)
        cov_lofo = model_lofo.covariance_
        precision_lofo = np.linalg.inv(cov_lofo)
    except np.linalg.LinAlgError:
        print(f"Skipping gene {g} due to singular covariance.")
        continue

    # 计算新的残差矩阵
    residual_lofo = precision_lofo - expected_matrix

    # 比较每条连接的残差变化
    for idx, (i, j) in enumerate(region_pairs):
        delta = residual_matrix[i, j] - residual_lofo[i, j]
        gene_contribution_df.iloc[idx, g] = delta



In [ ]:

# ---- Step 6: 添加连接信息列 ----
gene_contribution_df.insert(0, 'Region_Pair', [f'{i}-{j}' for i, j in region_pairs])
gene_contribution_df.to_csv(r"E:\aliyun_backup\muilt_disorders\15_Mitochondrial\Mitochondrial_contribution.csv", index=False)